In [45]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [69]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [70]:
train_dataset[0]

{'input_ids': [28866, 10, 7102, 55, 3, 23, 764, 640, 48, 8513, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 28866, 10, 3, 2, 11966, 834, 9269, 3155, 19542, 10, 2018, 55, 3, 10, 61, 1333, 6, 68, 27, 31, 162, 641, 608, 34, 5, 3, 10, 61, 19542, 10, 299, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [71]:
#random sampling
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [72]:
train_data.shape

(4000, 3)

## Data pre_processing

In [73]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",text) # lines
    text = re.sub(r"\s+"," ",text) # spaces
    text = re.sub(r"<.*?>"," ",text) # html tag
    text = text.strip().lower()
    return text

In [80]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summaury"] = val_data["summary"].apply(clean_data)

# Tokenize

In [81]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [82]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets = tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [83]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

# Working with our Model

In [84]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 348.74it/s]


In [85]:
import torch 

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ",device)
model.to(device)


device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [86]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps= 500
    
)

In [87]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = train_dataset,
    eval_dataset=val_dataset
)

In [88]:
#train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.645666,0.638313
2,0.396475,0.654592
3,0.373316,0.649336
4,0.362426,0.649391
5,0.355608,0.651951
6,0.351055,0.652713


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.23it/s]


TrainOutput(global_step=3000, training_loss=0.9140909372965494, metrics={'train_runtime': 2073.3548, 'train_samples_per_second': 11.575, 'train_steps_per_second': 1.447, 'total_flos': 3248203235328000.0, 'train_loss': 0.9140909372965494, 'epoch': 6.0})

In [22]:
type(train_dataset)

list

In [89]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:09<00:00,  9.60s/it]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [91]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 162.67it/s]


## Test the core logic for summarization

In [98]:
def summarize_dialogue(dialogue):
    model.to(device)
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,padding='max_length',
        max_length = 512,
        truncation = True,
        return_tensors = 'pt'
    )

    inputs.to(device)
    
    
    # generate the summary => token ids
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True
    )

    # token ids convert to text => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [99]:
test_dialogue = """Alice: Hey John, how was your weekend?

John: It was great! I went to Jaipur with my family. We visited some historical places and tried local food.

Alice: That sounds amazing! Which places did you visit?

John: We visited Hawa Mahal, Amber Fort, and City Palace. Amber Fort was my favorite because of its beautiful architecture.

Alice: Nice! How long did you stay there?

John: We stayed for three days. On the last day, we went shopping and bought some traditional clothes.

Alice: Sounds like a wonderful trip. Did you face any problems?

John: The weather was very hot, and we had to wait a long time for a taxi. But overall, it was a great experience.

Alice: I'm glad you enjoyed it! I hope I can visit Jaipur someday too."""

print(summarize_dialogue(test_dialogue))

john went to jaipur with his family. he visited historical places and tried local food.
